<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Greedy Tours and an Exact Reference

**Construct a feasible delivery tour and measure its quality against all tours of a small instance.**

Transfer the decision, feasibility, and objective roles to five-city delivery. The clock model and its constraints do not apply.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · The decision and fixed data

A vehicle visits cities $0,1,2,3,4$ once each and returns to its start. The fixed symmetric costs $C_{ij}$ are in **distance units**; they are not assumed to be map distances.

| From / to | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| $0$ | 0 | 2 | 9 | 10 | 7 |
| $1$ | 2 | 0 | 8 | 5 | 6 |
| $2$ | 9 | 8 | 0 | 4 | 3 |
| $3$ | 10 | 5 | 4 | 0 | 1 |
| $4$ | 7 | 6 | 3 | 1 | 0 |

The decision is the tour $x=(v_0,\ldots,v_5)$. Feasibility requires that $v_0,\ldots,v_4$ contain every city once and $v_5=v_0$. Its objective is total distance $f(x)=\sum_{k=0}^4C_{v_k,v_{k+1}}$. Reject an infeasible route before comparing scores.

For example, $(2,4,3,1,0,2)$ costs $3+1+5+2+9=20$. Omitting the return gives length $11$ but fails feasibility. The return edge belongs to the objective.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/complete_tour_return.png" alt="The return edge adds distance 9 to the partial route of length 11, making a complete tour of length 20." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · A local choice does not inspect the full tour

Nearest-neighbor construction chooses the smallest outgoing distance to an unvisited city, breaks ties by smaller city label, and returns after all visits. It commits to each choice. Here it is a **heuristic**: it produces a feasible tour without a general optimality guarantee.

Starting at city $2$, the choices are $2\to4\to3\to1\to0\to2$, with total $3+1+5+2+9=20$. The figure compares only the first outgoing choices; it does not compare complete tours.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/05-1_heuristic_optimization_greedy_and_local_search/assets/greedy_next_city.png" alt="From city 2, greedy construction chooses city 4 because its outgoing distance 3 is the smallest." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

| Construction start | $0$ | $1$ | $2$ | $3$ | $4$ |
|:---|---:|---:|---:|---:|---:|
| Final distance | $20$ | $22$ | $20$ | $24$ | $22$ |

The start is a search hyperparameter because no depot is imposed. Changing it changes construction choices, not the cost of a fixed tour. Rewriting a fixed cycle with a different start cannot improve it.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · An exact reference and a quality gap

Fix city $2$ as the written start and enumerate all $4!=24$ orders of the remaining cities, closing each route. This covers every feasible cycle; reverse orientations count twice because the matrix is symmetric. The minimum is $f^\star=20$, attained by $(2,4,3,1,0,2)$ and its reverse.

For a feasible tour,

$$
\operatorname{gap}(x)=100\,\frac{f(x)-f^\star}{f^\star}\%.
$$

Distances $20$, $21$, and $22$ have gaps $0\%$, $5\%$, and $10\%$. Enumeration certifies the global optimum for this finite instance. The greedy rule alone does not. With $n$ cities, enumeration uses $(n-1)!$ written tours: $24$ at $n=5$, but $362{,}880$ at $n=10$.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Interactive comparison

The start slider rebuilds the greedy tour; the edge-count slider shows its construction, including the return edge. An unfinished route is labeled incomplete and has no feasible-tour score. The exact reference is computed separately by enumerating all 24 written tours with city 2 fixed.

In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, PURPLE, GRAY = "#2878b5", "#168578", "#e78b24", "#8854a5", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Fixed travel distances; no map coordinates are assumed.
distances = np.array([[0,2,9,10,7], [2,0,8,5,6], [9,8,0,4,3],
                      [10,5,4,0,1], [7,6,3,1,0]])

def evaluate_tour(tour):
    tour = tuple(tour)
    feasible = (len(tour)==6 and set(tour[:-1])==set(range(5))
                and tour[-1]==tour[0])
    edges = [distances[a,b] for a,b in zip(tour[:-1], tour[1:])] if feasible else []
    return {"decision": tour, "feasible": feasible, "edges": edges,
            "objective": sum(edges) if feasible else None}

def greedy_tour(start):
    tour, unvisited = [start], set(range(5))-{start}
    while unvisited:
        city = min(unvisited, key=lambda candidate: (distances[tour[-1], candidate], candidate))
        tour.append(city)
        unvisited.remove(city)
    return evaluate_tour(tour+[start])

from itertools import permutations
exact_distance = min(evaluate_tour((2,)+order+(2,))["objective"]
                     for order in permutations([0,1,3,4]))

In [ ]:
figure, axis = plt.subplots(figsize=(5.6, 4.9))
figure.subplots_adjust(left=0.15, right=0.96, top=0.88, bottom=0.43)
bars = axis.bar(range(5), np.zeros(5), color=BLUE)
axis.set(xlabel="Chosen edge (last edge returns to start)", ylabel="Distance (units)",
         ylim=(0, 11), title="Greedy choices form a complete tour")
axis.grid(axis="y", alpha=0.25)
status = figure.text(0.15, 0.28, "", va="top")
start_slider = Slider(figure.add_axes([0.25, 0.12, 0.58, 0.035]), "start city",
                      0, 4, valinit=2, valstep=1, valfmt="%d", color=PURPLE)
edge_slider = Slider(figure.add_axes([0.25, 0.05, 0.58, 0.035]), "edges",
                     0, 5, valinit=5, valstep=1, valfmt="%d", color=ORANGE)

def update(_):
    result = greedy_tour(int(start_slider.val))
    count = int(edge_slider.val)
    tour, costs = result["decision"], result["edges"]
    for i, bar in enumerate(bars):
        bar.set_height(costs[i] if i<count else 0)
        bar.set_color(ORANGE if i==count-1 else BLUE)
    axis.set_xticks(range(5), [f"{a}→{b}" if i<count else "—"
                              for i,(a,b) in enumerate(zip(tour[:-1],tour[1:]))])
    visible = " → ".join(map(str, tour[:count+1]))
    if count==5:
        gap = 100*(result["objective"]-exact_distance)/exact_distance
        outcome = f"Feasible; distance = {result['objective']}; gap = {gap:.0f}%"
    else:
        outcome = f"Incomplete; traveled = {sum(costs[:count])}; no tour score"
    status.set_text(f"Route: {visible}\n{outcome}\nExact reference: {exact_distance} distance units")
    figure.canvas.draw_idle()

figure._sliders = [start_slider, edge_slider]
for slider in figure._sliders:
    slider.on_changed(update)
update(None)
plt.show()